# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule in plain words:
"A content item is likely declining if it has visibility (people see it in search) but it's not earning clicks for its position, and users who do visit aren't engaging with it."

That's the logic: seen but not clicked, visited but not engaged.

Reason codes:

* low_ctr_for_position — CTR is below what you'd expect given the content's
search position
* low_engagement_rate — engaged sessions are low relative to total sessions
* both_signals_weak — both CTR and engagement are underperforming

In [1]:

from datasets import load_dataset
from google.colab import userdata
import pandas as pd
from datetime import date

HF_TOKEN = userdata.get('HF_Token')

dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

train_split = dataset['train']

print("Loading January-April 2026 data...")
data_rows = []
batch_size = 100000
batch_count = 0

for batch in train_split.iter(batch_size=batch_size):
    batch_count += 1
    batch_df = pd.DataFrame(batch)

    # Ensure report_date is date object
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date

    # Filter for Jan-Apr 2026 and ga4_data_available = True (as per skill)
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]

    if len(data_batch) > 0:
        data_rows.append(data_batch)
        print(f"  Batch {batch_count}: {len(data_batch):,} rows")

df_4months = pd.concat(data_rows, ignore_index=True)
print(f"\n✓ Total: {len(df_4months):,} rows")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January-April 2026 data...
  Batch 200: 1,046 rows
  Batch 201: 702 rows
  Batch 202: 1,998 rows
  Batch 203: 1,008 rows
  Batch 204: 2,250 rows
  Batch 205: 1,597 rows
  Batch 206: 3,844 rows
  Batch 207: 351 rows
  Batch 208: 2,704 rows
  Batch 209: 1,323 rows
  Batch 210: 2,163 rows
  Batch 211: 1,730 rows
  Batch 212: 1,371 rows
  Batch 213: 1,721 rows
  Batch 214: 1,673 rows
  Batch 215: 594 rows
  Batch 216: 357 rows
  Batch 217: 1,035 rows
  Batch 218: 775 rows
  Batch 219: 1 rows
  Batch 220: 2,154 rows
  Batch 221: 59 rows
  Batch 222: 3,539 rows
  Batch 223: 2,723 rows
  Batch 224: 378 rows
  Batch 225: 896 rows
  Batch 226: 3,179 rows
  Batch 227: 1,314 rows
  Batch 228: 555 rows
  Batch 229: 2,693 rows
  Batch 230: 683 rows
  Batch 231: 2,031 rows
  Batch 232: 1,499 rows
  Batch 233: 1,270 rows
  Batch 234: 10 rows
  Batch 235: 500 rows
  Batch 236: 2,793 rows
  Batch 237: 931 rows
  Batch 238: 2,195 rows
  Batch 239: 1,294 rows
  Batch 240: 1,721 rows
  Batch 241: 

In [2]:
# ── Create is_declining_label ────────────────────────────────────────

# Monthly impressions per content item
df_4months['month'] = pd.to_datetime(df_4months['report_date']).dt.month

monthly = (
    df_4months[df_4months['month'].isin([3, 4])]
    .groupby(['content_hash_id', 'month'])['gsc_impressions']
    .sum()
    .unstack(fill_value=0)
)

monthly.columns = ['mar_impressions', 'apr_impressions']

# Label: 1 if April < 80% of March
monthly['is_declining_label'] = (
    monthly['apr_impressions'] < (0.8 * monthly['mar_impressions'])
).astype(int)

print(f"Labeled rows: {len(monthly):,}")
print(f"Declining: {monthly['is_declining_label'].mean():.1%}")

# Merge label back to daily data
df_4months = df_4months.merge(
    monthly[['is_declining_label']],
    on='content_hash_id',
    how='left'
)

Labeled rows: 142,628
Declining: 24.9%


In [3]:
# ── Signal 1: CTR-vs-Position (flag-linked: CTR-fix logic) ──────────

# Compute CTR, only where impressions > 0
df = df_4months.copy()
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, float('nan'))

# Position buckets
df['position_bucket'] = pd.cut(
    df['gsc_avg_position'],
    bins=[0, 3, 10, 20, 50, float('inf')],
    labels=['1-3', '4-10', '11-20', '21-50', '50+']
)

# Bucket table: decline rate by position bucket
signal_1 = (
    df[df['ctr'].notna()]
    .groupby('position_bucket', observed=True)
    .agg(
        n=('is_declining_label', 'count'),
        decline_rate=('is_declining_label', 'mean'),
        median_ctr=('ctr', 'median')
    )
    .round(3)
)

print("SIGNAL 1: CTR by Position Bucket")
print(signal_1)
print(f"\nn = {signal_1['n'].sum():,}")
print()

# ── Signal 2: Engagement Rate ───────────────────────────────────────

df['engagement_rate'] = (
    df['ga4_engaged_sessions'] / df['ga4_sessions'].replace(0, float('nan'))
)

df['engagement_bucket'] = pd.cut(
    df['engagement_rate'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']
)

signal_2 = (
    df[df['engagement_rate'].notna()]
    .groupby('engagement_bucket', observed=True)
    .agg(
        n=('is_declining_label', 'count'),
        decline_rate=('is_declining_label', 'mean'),
        median_engagement=('engagement_rate', 'median')
    )
    .round(3)
)

print("SIGNAL 2: Engagement Rate Bucket")
print(signal_2)
print(f"\nn = {signal_2['n'].sum():,}")

SIGNAL 1: CTR by Position Bucket
                      n  decline_rate  median_ctr
position_bucket                                  
1-3              112357         0.357       0.005
4-10             476339         0.325       0.004
11-20            213398         0.508       0.000
21-50            221589         0.583       0.000
50+               16409         0.522       0.000

n = 1,040,092

SIGNAL 2: Engagement Rate Bucket
                       n  decline_rate  median_engagement
engagement_bucket                                        
0-20%              19186         0.330              0.138
20-40%             17894         0.385              0.333
40-60%             17532         0.442              0.500
60-80%               829         0.481              0.667
80-100%            32619         0.468              1.000

n = 88,060


Signal 1 — CTR-vs-Position: CONFIRMED

Decline rate rises as position worsens: 32.5% at positions 4-10, jumps to 50.8% at 11-20, and 58.3% at 21-50. CTR drops to zero past position 10. Content that's visible but not clicked is more likely to decline. This is exactly what the CTR-fix flag logic predicts.

Signal 2 — Engagement Rate: OPPOSITE

Higher engagement rate actually correlates with higher decline rate (33% at 0-20% vs 46.8% at 80-100%). This is the reverse of what we expected. Also note the sample is much smaller — only 88,060 rows vs 1,040,092 for signal 1, meaning most rows lack engagement data.

This is a useful finding. The skill says "a clearly-explained negative is a win — it just saved your rule." Engagement rate would have weakened your baseline, not helped it.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# ── Encode the baseline rule ─────────────────────────────────────────

# Work at content level (one row per content item)
content = (
    df_4months[df_4months['gsc_impressions'] > 0]
    .groupby('content_hash_id')
    .agg(
        total_impressions=('gsc_impressions', 'sum'),
        total_clicks=('gsc_clicks', 'sum'),
        avg_position=('gsc_avg_position', 'mean'),
        is_declining_label=('is_declining_label', 'first')
    )
)

content['ctr'] = content['total_clicks'] / content['total_impressions']

# New: poor position + some visibility = high decline risk
visible = (content['total_impressions'] >= 100).astype(int)
content['score'] = content['avg_position'] * visible

# Reason code
content['reason_code'] = 'low_ctr_for_position'

# Action label
content['action'] = 'review_and_optimize'

# Rank by score descending
content = content.sort_values('score', ascending=False).reset_index()

import os
os.makedirs('work/outputs', exist_ok=True)

# Write ranked queue
output_cols = ['content_hash_id', 'score', 'reason_code', 'action',
               'total_impressions', 'ctr', 'avg_position', 'is_declining_label']
content[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Ranked queue: {len(content):,} rows")
print(f"Base rate: {content['is_declining_label'].mean():.3f}")

# Precision@K
def precision_at_k(df, k):
    return df.head(k)['is_declining_label'].mean()

for k in [10, 20, 50, 100]:
    print(f"Precision@{k}: {precision_at_k(content, k):.3f}")

Ranked queue: 96,613 rows
Base rate: 0.375
Precision@10: 0.500
Precision@20: 0.600
Precision@50: 0.542
Precision@100: 0.464


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# ── Top-20 Data ──────────────────────────────────────────────────────

top20 = content.head(20)[['content_hash_id', 'score', 'reason_code', 'action',
                           'total_impressions', 'ctr', 'avg_position', 'is_declining_label']]
top20.index = range(1, 21)
print(top20.to_string())


             content_hash_id      score           reason_code               action  total_impressions       ctr  avg_position  is_declining_label
1   content_d07ea12b5ba91c34  83.457307  low_ctr_for_position  review_and_optimize              139.0  0.000000     83.457307                 0.0
2   content_b49acf92cc1c8c7e  82.664151  low_ctr_for_position  review_and_optimize              265.0  0.000000     82.664151                 1.0
3   content_c26c91a74fe92a59  79.267379  low_ctr_for_position  review_and_optimize              170.0  0.011765     79.267379                 1.0
4   content_ec6f3d86a6c7cd7b  78.910569  low_ctr_for_position  review_and_optimize              123.0  0.000000     78.910569                 0.0
5   content_2392ac0360e7c84c  77.583534  low_ctr_for_position  review_and_optimize              530.0  0.000000     77.583534                 0.0
6   content_b0d0da82f52a25de  76.487452  low_ctr_for_position  review_and_optimize              116.0  0.000000     76.48745

## Top-20 Review

Precision@20: 0.600 | Base rate: 0.375 | 12 of 20 actually declining

| Rank | Action | Reason Code | Confidence | What Would Make It Wrong |
|------|--------|-------------|------------|--------------------------|
| 1 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Stable low performer, never had traffic to lose |
| 2 | review_and_optimize | low_ctr_for_position | HIGH — label=1, zero CTR, 265 impr | Would fail if impressions are bot-inflated |
| 3 | review_and_optimize | low_ctr_for_position | HIGH — label=1, near-zero CTR | Would fail if the tiny CTR signals early recovery |
| 4 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Lowest impressions in top 5; too small to meaningfully decline |
| 5 | review_and_optimize | low_ctr_for_position | LOW — label=0, highest impr in top 10 | High visibility but stable; position alone doesn't cause decline |
| 6 | review_and_optimize | low_ctr_for_position | MEDIUM — label=1, only 116 impr | Would fail if this content is being sunset intentionally |
| 7 | review_and_optimize | low_ctr_for_position | HIGH — label=1, near-zero CTR | Would fail if CTR is recovering in a trend the average hides |
| 8 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Barely above 100-impression threshold; never had enough traffic to decline |
| 9 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Stable at low baseline, not actively falling |
| 10 | review_and_optimize | low_ctr_for_position | HIGH — label=1, 372 impr, near-zero CTR | Real visibility with no clicks; hard to see how this isn't declining |
| 11 | review_and_optimize | low_ctr_for_position | HIGH — label=1, 250 impr | Would fail if the small CTR represents a positive trend |
| 12 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Low impressions, zero clicks but stable; not falling |
| 13 | review_and_optimize | low_ctr_for_position | HIGH — label=1, zero CTR, 178 impr | Would fail if impressions are from irrelevant queries |
| 14 | review_and_optimize | low_ctr_for_position | MEDIUM — label=1, only 102 impr | Barely visible; would fail if this is a new page still indexing |
| 15 | review_and_optimize | low_ctr_for_position | HIGH — label=1, zero CTR | Would fail if content is seasonal and expected to return |
| 16 | review_and_optimize | low_ctr_for_position | HIGH — label=1, 168 impr, zero CTR | Would fail if page was recently redirected |
| 17 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Tiny CTR but not declining; may be holding steady at low level |
| 18 | review_and_optimize | low_ctr_for_position | HIGH — label=1, near-zero CTR | Would fail if the engagement data tells a different story |
| 19 | review_and_optimize | low_ctr_for_position | LOW — label=0 | Zero clicks but stable; bad position is chronic, not worsening |
| 20 | review_and_optimize | low_ctr_for_position | HIGH — label=1, zero CTR, 186 impr | Would fail if decline is caused by external factor the rule can't see |

### Pattern in the misses
The 8 false positives (ranks 1, 4, 5, 8, 9, 12, 17, 19) share one trait: content stuck at bad positions with low impressions that was *never performing* — not declining. The rule cannot distinguish "falling" from "never rose." That's the gap a model should close.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks + Leakage Check

### Weak Picks

8 of 20 flagged items are NOT declining (ranks 1, 4, 5, 8, 9, 12, 17, 19). They share the same failure pattern:

**The rule confuses "never performed" with "declining."** These items sit at positions 72–83 with low impressions (100–530) and zero or near-zero CTR. They look bad — but they've always looked bad. Their March-to-April impressions didn't drop 20% because there wasn't much to drop.

The rule scores on current bad position, but decline requires *change over time*. A content item at position 80 with 110 impressions that stays flat is not declining — it's stagnant. The rule has no way to tell the difference.

**What a model could add:** A trend feature (e.g., impression change over a rolling window) would separate "falling from height" from "always on the floor."

### Leakage Check

| Check | Result |
|-------|--------|
| Does the score use `is_declining_label`? | NO — score uses `avg_position` and an impressions threshold only |
| Does the score use any April data to predict April decline? | The score aggregates across all months. However, the label compares March vs April. The score does not use the label or April-specific windows to compute itself — it ranks on position averaged across the full period |
| Does the score use any FlyRank product flags? | NO — no product-generated flags are used as inputs |
| Does the score use `gsc_sum_position` (excluded column)? | NO — uses `gsc_avg_position` which is in the approved feature list |
| Is any feature from the excluded list used? | NO — `gsc_data_available`, `ga4_data_available`, `gsc_sum_position` are all absent |
| Could a future window leak in? | The score uses `total_impressions` and `avg_position` aggregated over Jan–Apr. Since the label uses March vs April, the score does include April impressions in its average. This is a **minor concern** but not direct leakage — the score doesn't know the label boundary |

### Verdict

No direct leakage. One edge worth noting: the score averages position across all four months including April, while the label is defined by March-to-April change. This is not label leakage (the score never sees `is_declining_label`), but a stricter baseline would score only on Jan–March data. Worth flagging for the model phase.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.